In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

In [2]:
# imports + helpers reales reutilizables

from pprint import pprint

from enviroment_bj import (
    ACTION_ORDER,
    BlackjackConfig,
    BlackjackEnvironment,
    ObservationConfig,
    StartStateConfig,
)
from model.agents import (
    FeedForwardDoubleDQN,
    RecurrentDoubleDQN,
    DuelingRecurrentDoubleDQN,
)
from training import (
    build_trainer,
    train_model,
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
)


def make_env(
    observation_profile,
    *,
    seed=11,
    shoe=None,
    start_state=None,
    **config_overrides,
):
    observation = ObservationConfig.for_profile(observation_profile)
    config = BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=observation,
        **config_overrides,
    )
    env = BlackjackEnvironment(config=config, seed=seed, start_state=start_state)
    if shoe is not None:
        env.load_shoe(shoe, total_cards=len(shoe))
    return env


def make_pipeline_config(
    *,
    recurrent,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=None,
    warmup_size=None,
    sequence_length=4,
    min_sequence_length=2,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
):
    if batch_size is None:
        batch_size = 1 if recurrent else 4
    if warmup_size is None:
        warmup_size = 2 if recurrent else 8

    return TrainingPipelineConfig(
        trainer=TrainerConfig(
            total_epochs=total_epochs,
            env_steps_per_epoch=env_steps_per_epoch,
            train_frequency=1,
            updates_per_train_step=1,
            max_updates_per_epoch=max_updates_per_epoch,
            device="cpu",
            seed=13,
            reset_hidden_on_round_end=False,
            sequence_end_on_done=False,
            flush_partial_sequences_at_epoch_end=True,
        ),
        replay_buffer=ReplayBufferConfig(
            capacity=64,
            batch_size=batch_size,
            warmup_size=warmup_size,
            sequence_length=sequence_length,
            min_sequence_length=min_sequence_length,
        ),
        epsilon=EpsilonScheduleConfig(
            start=1.0,
            end=0.2,
            decay_steps=20,
            evaluation_epsilon=0.0,
        ),
        optimization=OptimizationConfig(
            optimizer="adam",
            learning_rate=1e-3,
            weight_decay=0.0,
            scheduler="none",
            gradient_clipping=True,
            max_grad_norm=5.0,
        ),
        target_update=TargetUpdateConfig(
            mode="hard",
            hard_update_interval=2,
            soft_tau=0.005,
        ),
        evaluation=EvaluationConfig(
            enabled=evaluation_enabled,
            every_n_epochs=1,
            num_rounds=4,
            max_decisions=200,
        ),
        checkpoints=CheckpointConfig(
            directory="tmp_unused",
            save_latest=False,
            save_best_eval=False,
            save_periodic=False,
            periodic_interval_updates=1000,
        ),
        prints=PrintConfig(enable=False),
    )


In [3]:
# instanciar 3 mesas típicas

env_ff = make_env(
    "minimal_basic_strategy",
    seed=11,
    shoe=["10", "6", "7", "10", "9", "5", "2", "10"],
)

env_rnn = make_env(
    "table_realistic_default",
    seed=21,
)

env_duel = make_env(
    "table_realistic_unknown_progress",
    seed=31,
    start_state=StartStateConfig(
        mode="unknown_progress",
        min_burned_rounds=2,
        max_burned_rounds=2,
        hide_reshuffle_progress_from_observation=True,
    ),
)

print(env_ff.config.observation.profile)
print(env_rnn.config.observation.profile)
print(env_duel.config.observation.profile)


minimal_basic_strategy
table_realistic_default
table_realistic_unknown_progress


In [4]:
# instanciar los 3 modelos en CPU

ff_model = FeedForwardDoubleDQN.from_profile("minimal_basic_strategy").cpu()
rnn_model = RecurrentDoubleDQN.from_profile("table_realistic_default", recurrent_type="gru").cpu()
duel_model = DuelingRecurrentDoubleDQN.from_profile(
    "table_realistic_unknown_progress",
    recurrent_type="lstm",
).cpu()

print("feedforward state_dim:", ff_model.state_dim)
print("recurrent state_dim:", rnn_model.state_dim)
print("dueling recurrent state_dim:", duel_model.state_dim)
print("acciones:", ACTION_ORDER)


feedforward state_dim: 40
recurrent state_dim: 1089
dueling recurrent state_dim: 1089
acciones: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')


In [5]:
# un solo backward real con FeedForwardDoubleDQN
# Hace warmup del buffer y luego exactamente un train_step()

ff_cfg = make_pipeline_config(
    recurrent=False,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=4,
    warmup_size=8,
    max_updates_per_epoch=1,
    evaluation_enabled=False,)

ff_trainer = build_trainer(env_ff, ff_model, pipeline_config=ff_cfg)

warmup_summary_ff = ff_trainer.warmup()
train_step_ff = ff_trainer.train_step()

print("warmup summary:")
pprint(warmup_summary_ff)

print("\ntrain_step metrics:")
pprint(train_step_ff["metrics"])

print("\nloss:", float(train_step_ff["loss"].item()))
print("buffer size:", len(ff_trainer.replay_buffer))
print("update_count:", ff_trainer.update_count)


warmup summary:
{'action_by_situation': {'hard_hands': {'hit': 3, 'stand': 3, 'surrender': 1},
                         'insurance_offers': {'stand': 1},
                         'pairs': {'surrender': 1},
                         'soft_hands': {'double': 1}},
 'action_frequencies': {'double': 0.125,
                        'hit': 0.375,
                        'insurance': 0.0,
                        'split': 0.0,
                        'stand': 0.375,
                        'surrender': 0.125},
 'ev_per_1000_hands': -250.0,
 'greedy_action_fraction': 0.0,
 'hands_completed': 8.0,
 'insurance_reward_total': 0.0,
 'loss_rate': 0.5,
 'push_rate': 0.125,
 'random_action_fraction': 1.0,
 'reward_per_hand': -0.25,
 'reward_per_round': -0.25,
 'rounds_completed': 8.0,
 'situation_counts': {'hard_hands': 7,
                      'insurance_offers': 1,
                      'pairs': 1,
                      'soft_hands': 1},
 'surrender_rate': 0.125,
 'table_counts': {'fresh_shoe|minimal_b

In [6]:
# un solo backward real con RecurrentDoubleDQN
# Igual idea, pero con buffer secuencial

rnn_cfg = make_pipeline_config(
    recurrent=True,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=1,
    warmup_size=2,
    sequence_length=4,
    min_sequence_length=2,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
)

rnn_trainer = build_trainer(env_rnn, rnn_model, pipeline_config=rnn_cfg)

warmup_summary_rnn = rnn_trainer.warmup()
train_step_rnn = rnn_trainer.train_step()

print("warmup summary:")
pprint(warmup_summary_rnn)

print("\ntrain_step metrics:")
pprint(train_step_rnn["metrics"])

print("\nloss:", float(train_step_rnn["loss"].item()))
print("buffer size:", len(rnn_trainer.replay_buffer))
print("update_count:", rnn_trainer.update_count)


warmup summary:
{'action_by_situation': {'hard_hands': {'double': 1, 'hit': 2, 'stand': 4},
                         'soft_hands': {'hit': 1}},
 'action_frequencies': {'double': 0.125,
                        'hit': 0.375,
                        'insurance': 0.0,
                        'split': 0.0,
                        'stand': 0.5,
                        'surrender': 0.0},
 'ev_per_1000_hands': -666.6666666666666,
 'greedy_action_fraction': 0.0,
 'hands_completed': 6.0,
 'insurance_reward_total': 0.0,
 'loss_rate': 0.6666666666666666,
 'push_rate': 0.3333333333333333,
 'random_action_fraction': 1.0,
 'reward_per_hand': -0.6666666666666666,
 'reward_per_round': -0.6666666666666666,
 'rounds_completed': 6.0,
 'situation_counts': {'hard_hands': 7, 'soft_hands': 1},
 'surrender_rate': 0.0,
 'table_counts': {'fresh_shoe|table_realistic_default': 8},
 'win_rate': 0.0}

train_step metrics:
{'buffer_size': 2.0,
 'epsilon': 0.6799999999999999,
 'grad_norm': 2.163269519805908,
 'learning

In [7]:
# un solo backward real con RecurrentDoubleDQN
# Igual idea, pero con buffer secuencial

rnn_cfg = make_pipeline_config(
    recurrent=True,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=1,
    warmup_size=2,
    sequence_length=4,
    min_sequence_length=2,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
)

rnn_trainer = build_trainer(env_rnn, rnn_model, pipeline_config=rnn_cfg)

warmup_summary_rnn = rnn_trainer.warmup()
train_step_rnn = rnn_trainer.train_step()

print("warmup summary:")
pprint(warmup_summary_rnn)

print("\ntrain_step metrics:")
pprint(train_step_rnn["metrics"])

print("\nloss:", float(train_step_rnn["loss"].item()))
print("buffer size:", len(rnn_trainer.replay_buffer))
print("update_count:", rnn_trainer.update_count)


warmup summary:
{'action_by_situation': {'hard_hands': {'hit': 4, 'stand': 4},
                         'insurance_offers': {'hit': 1}},
 'action_frequencies': {'double': 0.0,
                        'hit': 0.5,
                        'insurance': 0.0,
                        'split': 0.0,
                        'stand': 0.5,
                        'surrender': 0.0},
 'ev_per_1000_hands': -600.0,
 'greedy_action_fraction': 0.0,
 'hands_completed': 5.0,
 'insurance_reward_total': 0.0,
 'loss_rate': 0.8,
 'push_rate': 0.0,
 'random_action_fraction': 1.0,
 'reward_per_hand': -0.6,
 'reward_per_round': -0.6,
 'rounds_completed': 5.0,
 'situation_counts': {'hard_hands': 8, 'insurance_offers': 1},
 'surrender_rate': 0.0,
 'table_counts': {'fresh_shoe|table_realistic_default': 8},
 'win_rate': 0.2}

train_step metrics:
{'buffer_size': 2.0,
 'epsilon': 0.6799999999999999,
 'grad_norm': 1.2605111598968506,
 'learning_rate': 0.001,
 'loss': 0.32725098729133606,
 'max_abs_td_error': 1.30060923

In [8]:
#  un solo backward real con DuelingRecurrentDoubleDQN en unknown_progress

duel_cfg = make_pipeline_config(
    recurrent=True,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=1,
    warmup_size=2,
    sequence_length=4,
    min_sequence_length=2,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
)

duel_trainer = build_trainer(env_duel, duel_model, pipeline_config=duel_cfg)

warmup_summary_duel = duel_trainer.warmup()
train_step_duel = duel_trainer.train_step()

print("warmup summary:")
pprint(warmup_summary_duel)

print("\ntrain_step metrics:")
pprint(train_step_duel["metrics"])

print("\nloss:", float(train_step_duel["loss"].item()))
print("buffer size:", len(duel_trainer.replay_buffer))
print("update_count:", duel_trainer.update_count)


warmup summary:
{'action_by_situation': {'hard_hands': {'hit': 4, 'stand': 1, 'surrender': 1},
                         'soft_hands': {'hit': 1, 'stand': 1}},
 'action_frequencies': {'double': 0.0,
                        'hit': 0.625,
                        'insurance': 0.0,
                        'split': 0.0,
                        'stand': 0.25,
                        'surrender': 0.125},
 'ev_per_1000_hands': -687.5,
 'greedy_action_fraction': 0.0,
 'hands_completed': 8.0,
 'insurance_reward_total': 0.0,
 'loss_rate': 0.75,
 'push_rate': 0.0,
 'random_action_fraction': 1.0,
 'reward_per_hand': -0.6875,
 'reward_per_round': -0.6875,
 'rounds_completed': 8.0,
 'situation_counts': {'hard_hands': 6, 'soft_hands': 2},
 'surrender_rate': 0.125,
 'table_counts': {'unknown_progress|table_realistic_unknown_progress': 8},
 'win_rate': 0.125}

train_step metrics:
{'buffer_size': 2.0,
 'epsilon': 0.6799999999999999,
 'grad_norm': 2.014829158782959,
 'learning_rate': 0.001,
 'loss': 0.3582

In [9]:
# mini entrenamiento de 2 epochs con FeedForwardDoubleDQN
# Esto ya usa la API alta train_model(...), pero sigue siendo chiquito.

ff_model_2ep = FeedForwardDoubleDQN.from_profile("minimal_basic_strategy").cpu()
env_ff_2ep = make_env(
    "minimal_basic_strategy",
    seed=41,
    shoe=["10", "6", "7", "10", "9", "5", "2", "10", "4", "10"],
)

ff_cfg_2ep = make_pipeline_config(
    recurrent=False,
    total_epochs=2,
    env_steps_per_epoch=12,
    batch_size=4,
    warmup_size=8,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
)

ff_result_2ep = train_model(env_ff_2ep, ff_model_2ep, pipeline_config=ff_cfg_2ep)

print("epochs corridas:", len(ff_result_2ep["history"]))
print("update_count final:", ff_result_2ep["trainer"].update_count)
print("buffer final:", len(ff_result_2ep["trainer"].replay_buffer))

print("\nhistory resumido:")
for i, h in enumerate(ff_result_2ep["history"], start=1):
    print(
        f"epoch={i} | loss={h['loss']:.4f} | grad_norm={h['grad_norm']:.4f} | "
        f"buffer={h['buffer_size']:.0f} | epsilon={h['epsilon']:.4f}"
    )


epochs corridas: 2
update_count final: 2
buffer final: 31

history resumido:
epoch=1 | loss=0.1243 | grad_norm=0.6161 | buffer=20 | epsilon=0.2000
epoch=2 | loss=0.2555 | grad_norm=0.3384 | buffer=31 | epsilon=0.2000


In [10]:
# mini entrenamiento de 1 epoch con RecurrentDoubleDQN
# Puedes cambiar rnn_model por duel_model si quieres probar el dueling recurrent.

rnn_model_1ep = RecurrentDoubleDQN.from_profile("table_realistic_default", recurrent_type="gru").cpu()
env_rnn_1ep = make_env("table_realistic_default", seed=51)

rnn_cfg_1ep = make_pipeline_config(
    recurrent=True,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=1,
    warmup_size=2,
    sequence_length=4,
    min_sequence_length=2,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
)

rnn_result_1ep = train_model(env_rnn_1ep, rnn_model_1ep, pipeline_config=rnn_cfg_1ep)

print("epochs corridas:", len(rnn_result_1ep["history"]))
print("update_count final:", rnn_result_1ep["trainer"].update_count)
print("buffer final:", len(rnn_result_1ep["trainer"].replay_buffer))

print("\nsummary epoch 1:")
pprint(rnn_result_1ep["history"][0])


epochs corridas: 1
update_count final: 1
buffer final: 4

summary epoch 1:
{'action_by_situation': {'hard_hands': {'hit': 1, 'stand': 2, 'surrender': 4}},
 'action_frequencies': {'double': 0.0,
                        'hit': 0.14285714285714285,
                        'insurance': 0.0,
                        'split': 0.0,
                        'stand': 0.2857142857142857,
                        'surrender': 0.5714285714285714},
 'buffer_size': 4.0,
 'env_steps': 15.0,
 'epoch': 1.0,
 'epsilon': 0.3999999999999999,
 'ev_per_1000_hands': -428.57142857142856,
 'eval': None,
 'grad_norm': 1.66360342502594,
 'greedy_action_fraction': 0.8571428571428571,
 'hands_completed': 7.0,
 'insurance_reward_total': 0.0,
 'learning_rate': 0.001,
 'loss': 0.7264814376831055,
 'loss_rate': 0.2857142857142857,
 'max_abs_td_error': 1.9156213998794556,
 'mean_abs_td_error': 1.2264260053634644,
 'mean_q_pred': -0.0317012183368206,
 'mean_reward': -0.75,
 'mean_target': -0.75,
 'next_legal_fraction': 0.0

In [11]:
# mini entrenamiento de 1 epoch con DuelingRecurrentDoubleDQN en unknown_progress

duel_model_1ep = DuelingRecurrentDoubleDQN.from_profile(
    "table_realistic_unknown_progress",
    recurrent_type="lstm",
).cpu()

env_duel_1ep = make_env(
    "table_realistic_unknown_progress",
    seed=61,
    start_state=StartStateConfig(
        mode="unknown_progress",
        min_burned_rounds=2,
        max_burned_rounds=2,
        hide_reshuffle_progress_from_observation=True,
    ),
)

duel_cfg_1ep = make_pipeline_config(
    recurrent=True,
    total_epochs=1,
    env_steps_per_epoch=8,
    batch_size=1,
    warmup_size=2,
    sequence_length=4,
    min_sequence_length=2,
    max_updates_per_epoch=1,
    evaluation_enabled=False,
)

duel_result_1ep = train_model(env_duel_1ep, duel_model_1ep, pipeline_config=duel_cfg_1ep)

print("epochs corridas:", len(duel_result_1ep["history"]))
print("update_count final:", duel_result_1ep["trainer"].update_count)
print("buffer final:", len(duel_result_1ep["trainer"].replay_buffer))

print("\nsummary epoch 1:")
pprint(duel_result_1ep["history"][0])


epochs corridas: 1
update_count final: 1
buffer final: 4

summary epoch 1:
{'action_by_situation': {'hard_hands': {'hit': 1,
                                        'split': 1,
                                        'stand': 3,
                                        'surrender': 1},
                         'pairs': {'split': 1},
                         'post_split_states': {'stand': 2},
                         'soft_hands': {'hit': 1, 'stand': 1}},
 'action_frequencies': {'double': 0.0,
                        'hit': 0.25,
                        'insurance': 0.0,
                        'split': 0.125,
                        'stand': 0.5,
                        'surrender': 0.125},
 'buffer_size': 4.0,
 'env_steps': 16.0,
 'epoch': 1.0,
 'epsilon': 0.3599999999999999,
 'ev_per_1000_hands': -500.0,
 'eval': None,
 'grad_norm': 1.5284497737884521,
 'greedy_action_fraction': 0.75,
 'hands_completed': 5.0,
 'insurance_reward_total': 0.0,
 'learning_rate': 0.001,
 'loss': 0.71859574